# MBAI 448 | Week 6: VLM Sales Collateral Prototype

This notebook prototypes a Vision Language Model (VLM) system to generate sales collateral from property images. Each step follows a disciplined loop: **Plan → Validate → Execute → Check**.

## Step 1: Setup Environment

**Plan:** Install and import required libraries for running the Vision Language Model.

**Validate:** Confirm that the libraries are compatible and will run on your machine (use quantized models for local runs).

**Execute:** Install and import packages.

**Check:** Verify imports completed without errors.

In [ ]:
# If running in Colab, uncomment and run these lines:
# !pip install --quiet --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
# !pip install --quiet --no-deps cut_cross_entropy unsloth_zoo
# !pip install --quiet sentencepiece protobuf datasets huggingface_hub hf_transfer
# !pip install --quiet --no-deps unsloth

import torch
from PIL import Image
import os
from transformers import AutoTokenizer, AutoModelForVision2Seq
print('Imports successful!')

## Step 2: Load Vision Language Model

**Plan:** Load a quantized Vision Language Model (e.g., Qwen2.5-VL) and its tokenizer using 4-bit quantization for memory efficiency.

**Validate:** Confirm model card supports 4-bit quantization and understand its benefits (reduced memory, reasonable quality).

**Execute:** Load model and tokenizer.

**Check:** Confirm model and tokenizer loaded successfully. Review model size and memory usage.

In [ ]:
# Example: Replace with actual model if needed
model_name = 'Qwen/Qwen2.5-VL-7B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    device_map='auto',
    trust_remote_code=True,
    load_in_4bit=True
)
print('Model and tokenizer loaded!')

## Step 3: Sample Generation

**Plan:** Test the VLM with a sample image and prompt. Build helper functions for image loading and model inference.

**Validate:** Understand how VLMs combine visual and textual inputs.

**Execute:** Run inference on a sample image.

**Check:** Review the decoded output for relevance and quality.

In [ ]:
def load_image(image_path):
    return Image.open(image_path).convert('RGB')

def generate_caption(image, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    pixel_values = model.image_processor(image, return_tensors="pt").pixel_values.to(model.device)
    output = model.generate(**inputs, pixel_values=pixel_values, max_new_tokens=128)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Example usage (replace with actual image path):
sample_image_path = './images/sample.jpg'
if os.path.exists(sample_image_path):
    img = load_image(sample_image_path)
    prompt = "Describe this property for a rental listing."
    print(generate_caption(img, prompt))
else:
    print('Sample image not found. Please add an image to ./images/.')

## Step 4: Engineer Prompts

**Plan:** Create convenience functions for input processing and output generation. Test various prompts to build intuition for model behavior.

**Validate:** Functions should simplify the workflow and make iterative prompt testing easier.

**Execute:** Try different prompts and compare outputs.

**Check:** Evaluate performance across different image types.

In [ ]:
prompts = [
    "Describe this property for a rental listing.",
    "Highlight the best features of this space.",
    "What kind of tenant would enjoy living here?",
    "Write a short testimonial from a happy renter."
]

if os.path.exists(sample_image_path):
    img = load_image(sample_image_path)
    for p in prompts:
        print(f'Prompt: {p}')
        print(generate_caption(img, p))
        print('-'*40)
else:
    print('Sample image not found. Please add an image to ./images/.')

## Step 5: Generalize Your Prompts

**Plan:** Develop prompts specific to the sales collateral use case—describing spaces, highlighting possibilities, and generating testimonials.

**Validate:** Ensure prompts are grounded in what the model can actually see.

**Execute:** List and test prompts for each use case.

**Check:** Review outputs for each prompt type.

In [ ]:
general_prompts = {
    'description': "Describe the main features of this property.",
    'possibilities': "Suggest creative uses for this space.",
    'testimonial': "Write a short testimonial from a satisfied renter."
}

if os.path.exists(sample_image_path):
    img = load_image(sample_image_path)
    for label, p in general_prompts.items():
        print(f'{label.title()} Prompt: {p}')
        print(generate_caption(img, p))
        print('-'*40)
else:
    print('Sample image not found. Please add an image to ./images/.')

## Step 6: Test Your Prompts

**Plan:** Systematically test prompts against multiple property images to assess reliability and quality.

**Validate:** Build a sampling function to efficiently test across the image dataset.

**Execute:** Run the function and review outputs.

**Check:** Which prompts produce the most reliable, useful content?

In [ ]:
def batch_generate(image_dir, prompts):
    results = {}
    for fname in os.listdir(image_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(image_dir, fname)
            img = load_image(img_path)
            results[fname] = {}
            for label, p in prompts.items():
                results[fname][label] = generate_caption(img, p)
    return results

image_dir = './images/'
if os.path.exists(image_dir):
    outputs = batch_generate(image_dir, general_prompts)
    for fname, out in outputs.items():
        print(f'Image: {fname}')
        for label, text in out.items():
            print(f'{label.title()}: {text}')
        print('-'*40)
else:
    print('Image directory not found.')

## Step 7: Structure Prompts into Workflow

**Plan:** Combine effective prompts into a cohesive workflow that generates a complete sales collateral document from a single image.

**Validate:** The workflow should output image, description, possibilities, and testimonial.

**Execute:** Run the workflow for a sample image.

**Check:** Evaluate the completeness and reliability of the generated collateral.

In [ ]:
def generate_sales_collateral(image_path, prompts):
    img = load_image(image_path)
    collateral = {}
    for label, p in prompts.items():
        collateral[label] = generate_caption(img, p)
    return collateral

if os.path.exists(sample_image_path):
    doc = generate_sales_collateral(sample_image_path, general_prompts)
    print(f'Sales Collateral for {sample_image_path}:')
    for section, text in doc.items():
        print(f'{section.title()}: {text}')
else:
    print('Sample image not found. Please add an image to ./images/.')